# 🛡️ Detector UEBA: Detecção de Anomalias em Logs de Autenticação
### Dissertação de Mestrado Profissional em Engenharia Elétrica (UnB / FT / ENE)
* **Autor:** Allysson de Oliveira Noronha
* **Orientador:** Prof. Dr. Robson de Oliveira Albuquerque
* **Repositório:** [https://github.com/allyssonnor/anti_Detector](https://github.com/allyssonnor/anti_Detector)

---

### 🎯 Propósito deste Demonstrador
Este notebook foi projetado para que a **Banca Examinadora** possa reproduzir e auditar com facilidade o pipeline experimental:
1. **Auditoria de Causalidade (`pytest`):** Garantia matemática de ausência de vazamento temporal (*data leakage*).
2. **Treinamento Comparativo (Generalistas vs Especialistas):**
   * **Generalistas (`EF_Amplo`):** Treinados em tráfego multimodal contendo múltiplos ataques simultâneos (Força Bruta, Fora de Hora, Movimentação Lateral).
   * **Especialistas (`EF_Bruteforce`):** Treinados exclusivamente sobre campanhas focadas de Força Bruta e Password Spraying.
3. **Inferência Cruzada e Generalização (*Zero-Shot Domain Transfer*):**
   * Especialista avaliado contra o ataque isolado de força bruta (`test_bruteforce_42.jsonl`).
   * Especialista e Generalista avaliados contra o cenário multimodal complexo (`seed_1_sample.jsonl`).
4. **Dashboard Forense e Métricas:** Visualização imediata de Matriz de Confusão, PR-AUC, ROC-AUC, Lift e detalhamento dos alertas emitidos.

---

### 📦 Datasets Disponíveis no Repositório (`data/`)
1. **`data/ef_amplo.jsonl`** (~6 MB, 20.355 eventos) — Cenário EvidenceForge com múltiplos ataques integrados.
2. **`data/ef_bruteforce.jsonl`** (~5.9 MB, 19.032 eventos) — Cenário EvidenceForge especialista em ataques de Força Bruta.
3. **`data/test_bruteforce_42.jsonl`** (~50.4 MB, 122.239 eventos) — Cenário de teste confirmatório de transferência com ataque de força bruta injetado.
4. **`data/seed_1_sample.jsonl`** (~7.0 MB, 16.914 eventos) — Amostra com integridade de sequências temporais do dataset massivo multimodal.

## 1. ⚙️ Preparação do Ambiente e Clonagem do Repositório

In [ ]:
# 1. Clonagem resiliente do repositório (diretório fixo /content/anti_Detector)
import os

%cd /content
if not os.path.isdir("/content/anti_Detector"):
    !git clone https://github.com/allyssonnor/anti_Detector.git
%cd /content/anti_Detector
!git pull

# 2. Instalação das dependências necessárias
!pip install -q -r requirements.txt


## 2. 🧪 Auditoria de Integridade e Causalidade Temporal
Execução da suíte de testes de causalidade para comprovar que nenhuma feature utiliza informação futura no instante $t$.

In [ ]:
!python -m pytest tests/test_causality.py -v

## 3. 🏋️ Treinamento e Calibração Dinâmica de Limiares
Execução do treinamento cronológico (70% treino, 20% validação, 10% teste) dos modelos **Generalistas** (`EF_Amplo`) e **Especialistas** (`EF_Bruteforce`).
Para cada dataset, são treinados e calibrados com limiar dinâmico $F_2$:
* **RandomForest** (classificador comitê supervisionado)
* **MLP_Single** (Autoencoder neural baseado em erro de auto-reconstrução $X \to X$)


In [ ]:
## 4. 📊 Tabela Consolidada de Resultados de Validação e Teste Misto

import pandas as pd
import glob

# Busca resiliente pelo consolidado master_results_table.csv
res_csv = glob.glob("**/master_results_table.csv", recursive=True)
if not res_csv:
    res_csv = glob.glob("results/**/master_results*.csv", recursive=True)

if res_csv:
    df_res = pd.read_csv(res_csv[0])
    cols = [c for c in ["model_preset", "dataset", "precision", "recall", "f1_score", "f2_score", "pr_auc", "roc_auc", "lift_pr", "tp", "fp", "fn", "tn"] if c in df_res.columns]
    print("\n📈 TABELA CONSOLIDADA DE RESULTADOS (Modelos Generalistas e Especialistas):")
    display(df_res[cols].drop_duplicates())
else:
    print("Nenhum resultado consolidado encontrado. Verifique se o treino na Célula 3 concluiu.")


In [ ]:
## 5. 🔍 Inferência Cruzada Multicenário (Zero-Shot Transfer)
Nesta etapa, aplicamos os modelos treinados (com os limiares calibrados na validação) sobre datasets nunca vistos para avaliar generalização e capacidade de discriminação:

| Cenário | Modelo Utilizado | Dataset Alvo | Objetivo de Avaliação |
| :--- | :--- | :--- | :--- |
| **A (Recomendado)** | **Especialista Bruteforce (RF)** | `data/test_bruteforce_42.jsonl` | Avaliar se o especialista detecta o ataque isolado de força bruta inédito com alta precisão e recall. |
| **B** | **Especialista Bruteforce (RF)** | `data/seed_1_sample.jsonl` | Avaliar a seletividade: detectar as instâncias de força bruta sem disparar em outras anomalias. |
| **C** | **Especialista Bruteforce (RF)** | `data/ef_amplo.jsonl` | Avaliar o especialista contra o tráfego multimodal da mesma família EvidenceForge. |
| **D** | **Generalista (RF)** | `data/seed_1_sample.jsonl` | Avaliar a transferência do modelo multimodal para outro ambiente corporativo. |

# ======================================================================
# CONFIGURAÇÃO DO CENÁRIO DE INFERÊNCIA CRUZADA
# ======================================================================
# Escolha um dos cenários abaixo descomentando a linha correspondente:

# --- Cenário A: Especialista em Bruteforce -> test_bruteforce_42 (Ataque isolado)
MODEL_DIR = "results/demo_colab/full/run_seed42/EF_Bruteforce_Especialista_RandomForest_global_20"
DATA_TARGET = "data/test_bruteforce_42.jsonl"

# --- Cenário B: Especialista em Bruteforce -> seed_1_sample (Cenário multimodal)
# MODEL_DIR = "results/demo_colab/full/run_seed42/EF_Bruteforce_Especialista_RandomForest_global_20"
# DATA_TARGET = "data/seed_1_sample.jsonl"

# --- Cenário C: Especialista em Bruteforce -> ef_amplo (EvidenceForge multimodal)
# MODEL_DIR = "results/demo_colab/full/run_seed42/EF_Bruteforce_Especialista_RandomForest_global_20"
# DATA_TARGET = "data/ef_amplo.jsonl"

# --- Cenário D: Generalista -> seed_1_sample (Transferência de domínio ampla)
# MODEL_DIR = "results/demo_colab/full/run_seed42/EF_Amplo_Generalista_RandomForest_global_20"
# DATA_TARGET = "data/seed_1_sample.jsonl"

print(f"🚀 Executando Inferência:\n🔹 Modelo: {MODEL_DIR}\n🔹 Dataset: {DATA_TARGET}\n")
!python scripts/inferir_usual.py --config config/demo_colab.yaml --data $DATA_TARGET --model_dir $MODEL_DIR


In [ ]:
# Escolha do modelo e do dataset de teste para inferência cruzada:
MODEL_DIR = "results/demo_colab/full/run_seed42/Synthetic_Chronological_RandomForest_global_20"
DATA_TARGET = "data/test_bruteforce_42.jsonl"  # Alterne para data/seed_1_sample.jsonl ou data/ef_bruteforce.jsonl

!python scripts/inferir_usual.py --config config/demo_colab.yaml --data $DATA_TARGET --model_dir $MODEL_DIR


from IPython.display import Image, display
import glob
import os
import pandas as pd

# 1. Exibição do Dashboard Forense gerado para o modelo e dataset avaliados
dataset_tag = os.path.splitext(os.path.basename(DATA_TARGET))[0]
dashboard_img = f"{MODEL_DIR}/dashboard_{dataset_tag}.png"

if os.path.exists(dashboard_img):
    print(f"📊 Dashboard Forense ({dataset_tag}):")
    display(Image(dashboard_img))
else:
    # Fallback caso procure por qualquer dashboard no diretório
    dashboards = glob.glob(f"{MODEL_DIR}/dashboard_*.png")
    if dashboards:
        display(Image(dashboards[0]))
    else:
        print(f"Dashboard não encontrado em: {MODEL_DIR}")

# 2. Amostra dos alertas disparados com atributos de auditoria
eventos_csv = f"{MODEL_DIR}/inferencia_{dataset_tag}.csv"
if os.path.exists(eventos_csv):
    df_inf = pd.read_csv(eventos_csv)
    print(f"\n🛡️ Amostra de Alertas Disparados (Total no dataset: {len(df_inf)} eventos):")
    alertas = df_inf[df_inf["is_alert"] == 1] if "is_alert" in df_inf.columns else df_inf.head(10)
    cols_show = [c for c in ["UserID", "Time", "IpAddress", "LogHost", "anomaly_score", "is_alert", "Is_Anomaly", "AnomalyType"] if c in df_inf.columns]
    display(alertas[cols_show].head(10))


In [ ]:
from IPython.display import Image, display
import glob
import os
import pandas as pd

# Busca pelo dashboard gerado na inferência mais recente
dashboards = glob.glob(f"{MODEL_DIR}/dashboard_*.png")
if dashboards:
    display(Image(dashboards[0]))
else:
    print(f"Dashboard não encontrado em: {MODEL_DIR}")

# Amostra das detecções e anomalias identificadas
eventos_csvs = glob.glob(f"{MODEL_DIR}/inferencia_*.csv")
if eventos_csvs:
    df_inf = pd.read_csv(eventos_csvs[0])
    print("\n🛡️ Top 10 Alertas Disparados com Detalhes Comportamentais:")
    alertas = df_inf[df_inf["alert"] == 1] if "alert" in df_inf.columns else df_inf.head(10)
    cols_show = [c for c in ["UserID", "Time", "IpAddress", "LogHost", "anomaly_score", "threshold", "alert", "Is_Anomaly", "AnomalyType"] if c in df_inf.columns]
    display(alertas[cols_show].head(10))
